# Algenta — Google Colab

Run live contract discovery, low-token governed dataset discovery, exact queries, and probabilistic simulations from your notebook.

**What you get:**
- Live machine-readable contract with `GET /v1/meta/contract`, plus `/openapi.json` fallback for older self-hosted nodes
- Dataset discovery with `GET /v1/data?search=...&compact=1`
- Low-token dataset summary with `GET /v1/data/{dataset_id}/summary`
- Governed exact queries through `/v1/query`, `/v1/query/batch`, and `/v1/query/sql-report`
- Monte Carlo simulations through `/v1/simulate` when you need scenario analysis

**Get your API key:** Use https://app.algenta.ai/dashboard/api-keys only in Cloud Managed. In `self_hosted` and `air_gapped`, use the API key provisioned by your self-hosted operator deployment.\n\n**Base URL:** Replace `https://your-algenta-base-url.example` with Cloud Managed `https://api.algenta.ai` or your self-hosted base URL before running the notebook. Private profiles do not silently fall back to Algenta cloud.

In [ ]:
# Install dependencies
!pip install requests pandas matplotlib seaborn -q

In [ ]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0B0D12'
matplotlib.rcParams['axes.facecolor'] = '#11131A'
matplotlib.rcParams['text.color'] = '#E6E8EC'
matplotlib.rcParams['axes.labelcolor'] = '#8A90A2'
matplotlib.rcParams['xtick.color'] = '#8A90A2'
matplotlib.rcParams['ytick.color'] = '#8A90A2'

# ── Configuration ──────────────────────────────────────────────────────────
API_KEY = "<YOUR_ALGENTA_API_KEY>"  # Cloud Managed only; use a self-hosted-issued key in private profiles
BASE_URL = "https://your-algenta-base-url.example"  # Replace with Cloud Managed https://api.algenta.ai or your self-hosted base URL

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

## Example 0: Discover a dataset and query exact metrics

Use the low-token governed data path first:
1. `GET /v1/meta/contract` (or `/openapi.json` plus `x-primary-data-query-contract` on older self-hosted nodes)
2. `GET /v1/data?search=...&compact=1`
3. `GET /v1/data/{dataset_id}/summary`
4. `POST /v1/query/batch`

Use `POST /v1/query/sql-report` only when you need a wide read-only rowset.

In [ ]:
def load_contract() -> dict:
    contract = requests.get(f"{BASE_URL}/v1/meta/contract", headers=headers, timeout=30)
    if contract.status_code != 404:
        contract.raise_for_status()
        return contract.json()

    openapi = requests.get(f"{BASE_URL}/openapi.json", headers=headers, timeout=30)
    openapi.raise_for_status()
    schema = openapi.json()
    extension = schema.get("x-primary-data-query-contract")
    if not isinstance(extension, dict):
        raise RuntimeError("This service does not publish x-primary-data-query-contract in /openapi.json.")
    return {
        "brand": schema.get("info", {}).get("title", "Algenta"),
        "primary_data_query_contract": extension,
    }

contract_payload = load_contract()
print("Contract endpoint:", contract_payload["primary_data_query_contract"]["api"]["contract_endpoint"])

listed = requests.get(
    f"{BASE_URL}/v1/data?search=orders&compact=1&limit=5",
    headers=headers,
)
listed.raise_for_status()
listed_payload = listed.json()
datasets = listed_payload.get('datasets', [])
if not datasets:
    raise RuntimeError("No dataset matched search 'orders'. Change the search term for your workspace.")

dataset_id = datasets[0]['dataset_id']
summary = requests.get(f"{BASE_URL}/v1/data/{dataset_id}/summary", headers=headers)
summary.raise_for_status()
summary_payload = summary.json()

batch_payload = {
    "defaults": {"dataset_id": dataset_id},
    "queries": [
        {
            "key": "monthly_completed_orders",
            "request": {
                "metric": {"role": "derived_measure", "hint": "completed_order_count"},
                "aggregation": "sum",
                "group_by": ["order_month"],
                "limit": 12,
                "order": "desc",
            },
        },
        {
            "key": "average_order_value",
            "request": {
                "metric": {"role": "base_measure", "hint": "gross_revenue"},
                "aggregation": "avg",
                "group_by": ["order_month"],
                "limit": 12,
                "order": "desc",
            },
        },
    ],
}

batch = requests.post(f"{BASE_URL}/v1/query/batch", headers=headers, json=batch_payload)
batch.raise_for_status()
batch_result = batch.json()

print(f"Dataset        : {summary_payload['name']} ({dataset_id})")
print(f"Matched Total  : {listed_payload.get('matched_total')}")
print(f"Request ID     : {batch_result.get('request_id')}")
for item in batch_result.get('results', []):
    print(f"\n{item['key']} ->")
    if item.get('error'):
        print(item['error'])
    else:
        print(item.get('metadata', {}))
        print(item.get('data', {}).get('result'))

## Example 1: Simple Business Decision (Auto Mode)

Should we launch this product? Define variable ranges, get a structured recommendation.

In [ ]:
payload = {
    "mode": "auto",
    "runs": 50000,
    "seed": 42,
    "scenario": {
        "name": "Product Launch Decision",
        "variables": {
            "revenue":     {"low": 80000,  "high": 250000},
            "cost":        {"low": 40000,  "high": 120000},
            "growth_rate": {"low": 0.05,   "high": 0.35},
            "churn_rate":  {"low": 0.02,   "high": 0.12},
        },
        "objective": "maximize_net_value"
    }
}

response = requests.post(f"{BASE_URL}/v1/simulate", headers=headers, json=payload)
response.raise_for_status()
result = response.json()

print(f"Recommendation : {result['recommended_action'].upper()}")
print(f"Confidence     : {result['confidence']:.1%}")
print(f"Expected Value : ${result['metrics']['expected_value']:,.0f}")
print(f"Prob of Loss   : {result['metrics']['probability_of_loss']:.1%}")
print(f"VaR 95%        : ${result['metrics']['var_95']:,.0f}")
print(f"Scenarios Run  : {result['scenarios_run']:,}")
print(f"\nRationale: {result['rationale']}")
print(f"\nResult Hash (SHA-256): {result['result_hash']}")

## Example 2: Expert Mode — Full Distribution Control

In [ ]:
expert_payload = {
    "mode": "expert",
    "runs": 100000,
    "seed": 99,
    "simulation_model": "lhs",  # Latin Hypercube Sampling for better coverage
    "simulation": {
        "variables": [
            {
                "name": "arr",
                "distribution": "lognormal",
                "params": {"mean": 14.0, "std": 0.4},
                "unit": "USD",
                "description": "Annual Recurring Revenue"
            },
            {
                "name": "churn",
                "distribution": "beta",
                "params": {"alpha": 2, "beta": 22},
                "unit": "%/month",
                "description": "Monthly churn rate"
            },
            {
                "name": "ltv_multiple",
                "distribution": "triangular",
                "params": {"low": 2.5, "mode": 4.0, "high": 7.0},
                "unit": "x",
                "description": "LTV/CAC multiple"
            }
        ],
        "objective_function": "arr * ltv_multiple * (1 - churn * 12)",
        "output_labels": ["Risk-Adjusted ARR Value"],
        "output_units": ["USD"],
        "scoring": {"expected_value": 0.6, "downside_risk": 0.4}
    }
}

r2 = requests.post(f"{BASE_URL}/v1/simulate", headers=headers, json=expert_payload)
r2.raise_for_status()
result2 = r2.json()

print(f"Recommendation : {result2['recommended_action'].upper()}")
print(f"Confidence     : {result2['confidence']:.1%}")
print(f"Expected Value : ${result2['metrics']['expected_value']:,.0f}")
print(f"Model Used     : {result2.get('simulation_model', 'lhs')}")

## Visualize Percentile Distribution

In [ ]:
p = result['percentiles']
metrics = result['metrics']

labels = ['P5', 'P25', 'P50', 'P75', 'P95']
values = [p.get('p5',0), p.get('p25',0), p.get('p50',0), p.get('p75',0), p.get('p95',0)]
colors = ['#EF4444' if v < 0 else '#6B42FC' for v in values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — percentiles
axes[0].bar(labels, values, color=colors, edgecolor='#1F2430')
axes[0].axhline(0, color='#2A3142', linewidth=1, linestyle='--')
axes[0].set_title('Outcome Distribution by Percentile', color='#E6E8EC', pad=12)
axes[0].set_ylabel('Net Value ($)', color='#8A90A2')
axes[0].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Metrics summary
metric_labels = ['Expected\nValue', 'Std\nDeviation', 'Prob of\nLoss (%)']
metric_values = [
    metrics.get('expected_value', 0),
    metrics.get('std_deviation', 0),
    metrics.get('probability_of_loss', 0) * 100
]
metric_colors = ['#00D395', '#6B42FC', '#EF4444']

bars = axes[1].bar(metric_labels, metric_values, color=metric_colors, edgecolor='#1F2430', width=0.5)
axes[1].set_title('Key Risk Metrics', color='#E6E8EC', pad=12)

for bar, val in zip(bars, metric_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + abs(bar.get_height())*0.02,
                f'{val:,.1f}', ha='center', va='bottom', color='#E6E8EC', fontsize=10)

plt.tight_layout()
plt.savefig('algenta_simulation_result.png', dpi=150, bbox_inches='tight',
            facecolor='#0B0D12', edgecolor='none')
plt.show()
print("Chart saved to algenta_simulation_result.png")

## Use Templates (18 Industry Domains)

In [ ]:
# List all available templates
templates_resp = requests.get(f"{BASE_URL}/v1/simulate/templates", headers=headers)
templates_resp.raise_for_status()
templates = templates_resp.json()['templates']

df = pd.DataFrame([{
    'Name': t['name'],
    'Domain': t['domain'],
    'Industry': t['industry'],
    'Variables': t['n_variables'],
    'Tags': ', '.join(t['tags'][:3]),
} for t in templates])

print(f"Available templates: {len(templates)}")
display(df)

In [ ]:
# Run a specific template by ID
template_id = templates[0]['id']  # First template
template_name = templates[0]['name']

template_detail = requests.get(
    f"{BASE_URL}/v1/simulate/templates/{template_id}?runs=50000",
    headers=headers
).json()

# Run it
template_result = requests.post(
    f"{BASE_URL}/v1/simulate",
    headers=headers,
    json=template_detail['payload']
).json()

print(f"Template: {template_name}")
print(f"Result  : {template_result.get('recommended_action','').upper()} @ {template_result.get('confidence',0):.1%} confidence")